# Earnings Call Prediction - Evaluation Notebook
This notebook loads a trained FinBERT regression checkpoint, evaluates it on the same 80/20 split used during training, and produces metrics and plots. Because the dataset has only 173 transcripts, the exact split matters.  The README reports the best observed run as a best-case result and discusses why performance is unstable across runs.


I moved the plots from my original notebook to this notebook in order to display the results 

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

from earnings_transcript_predictor.models import EarningsModel
from earnings_transcript_predictor.dataset import load_records, EarningsDataset
from earnings_transcript_predictor.evaluation import regression_metrics, signal, signal_accuracy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cpu


Load data and model 

In [ ]:
DATA_DIR = 'notebooks/data/transcripts'
MODEL_PATH = 'checkpoints/best_model.pt'
SEED = 42

records = load_records(DATA_DIR)
dataset = EarningsDataset(records, max_length=512)

# Reproduce the train/test split for the selected seed. Change SEED to evaluate a different split.
n_test = max(1, int(len(dataset) * 0.2))
n_train = len(dataset) - n_test
generator = torch.Generator().manual_seed(SEED)
train_ds, test_ds = random_split(dataset, [n_train, n_test], generator=generator)
test_loader = DataLoader(test_ds, batch_size=8)
print(f'Train set: {n_train} | Test set: {n_test} | Seed: {SEED}')

model = EarningsModel().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print('Model loaded.')


Make predictions

In [ ]:
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        preds = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['label'].numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
errors = all_preds - all_labels
metrics = regression_metrics(all_preds, all_labels)

print(f"MAE:  {metrics['mae']:.2f}%")
print(f"RMSE: {metrics['rmse']:.2f}%")
print(f"Bias: {metrics['bias']:.2f}%")
print(f"Directional accuracy: {metrics['directional_accuracy']:.2f}")


Buy/Sell/Hold Signal Accuracy

In [ ]:
THRESHOLD = 0.5
rows = signal_accuracy(all_preds, all_labels, [THRESHOLD])
row = rows[0]
print(f"Signal Accuracy (+/-{THRESHOLD}%): {row['correct']}/{row['total']} = {row['accuracy']:.2f}")

print('\nAll predictions:')
print(f'{"Predicted":>10} {"Actual":>10} {"Pred Sig":>8} {"Actual":>8} {"Correct?":>10}')
print('-' * 55)
for pred, actual in zip(all_preds, all_labels):
    pred_signal = signal(pred, THRESHOLD)
    actual_signal = signal(actual, THRESHOLD)
    match = 'yes' if pred_signal == actual_signal else 'no'
    print(f'{pred:>10.2f}% {actual:>10.2f}% {pred_signal:>8} {actual_signal:>8} {match:>10}')


Signal Accuracy by Threshold

In [ ]:
thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
signal_rows = signal_accuracy(all_preds, all_labels, thresholds)
accuracies = [row['accuracy'] for row in signal_rows]
for row in signal_rows:
    print(f"Threshold +/-{row['threshold']}%: {row['correct']}/{row['total']} = {row['accuracy']:.2f}")

plt.figure(figsize=(8, 5))
plt.bar(thresholds, accuracies, color='steelblue', alpha=0.7, width=0.3, edgecolor='black')
plt.axhline(0.5, color='red', linestyle='--', linewidth=2, label='Random direction baseline')
plt.xlabel('Buy/Sell/Hold Threshold (%)', fontsize=11)
plt.ylabel('Signal Accuracy', fontsize=11)
plt.title('Signal Accuracy by Threshold', fontsize=12, fontweight='bold')
plt.legend()
plt.ylim(0, 1)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('signal_accuracy.png', dpi=150)
plt.show()


Prediction Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predicted vs Actual scatter
axes[0].scatter(all_labels, all_preds, alpha=0.7, color='steelblue', s=80, edgecolors='black', linewidth=0.5)
axes[0].plot([-10, 10], [-10, 10], 'r--', alpha=0.5, label='Perfect prediction')
axes[0].axhline(0, color='black', linewidth=0.5, alpha=0.3)
axes[0].axvline(0, color='black', linewidth=0.5, alpha=0.3)
axes[0].set_xlabel('Actual Return (%)', fontsize=11)
axes[0].set_ylabel('Predicted Return (%)', fontsize=11)
axes[0].set_title('Predicted vs Actual Returns', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Side by side bar
x = np.arange(len(all_labels))
axes[1].bar(x - 0.175, all_labels, 0.35, label='Actual', color='steelblue', alpha=0.7)
axes[1].bar(x + 0.175, all_preds, 0.35, label='Predicted', color='orange', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Test Sample', fontsize=11)
axes[1].set_ylabel('Return (%)', fontsize=11)
axes[1].set_title('Actual vs Predicted Per Sample', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

# Error distribution
axes[2].hist(errors, bins=15, color='steelblue', alpha=0.7, edgecolor='black')
axes[2].axvline(0, color='red', linestyle='--', label='Zero error')
axes[2].axvline(np.mean(errors), color='orange', linestyle='--', label=f'Mean: {np.mean(errors):.2f}%')
axes[2].set_xlabel('Prediction Error (%)', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_title('Distribution of Prediction Errors', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_analysis.png', dpi=150)
plt.show()

print(f'MAE:  {np.mean(np.abs(errors)):.2f}%')
print(f'RMSE: {np.sqrt(np.mean(errors**2)):.2f}%')
print(f'Bias: {np.mean(errors):.2f}%')

Summary Statistics

In [ ]:
best_row = max(signal_rows, key=lambda row: row['accuracy'])

print('=' * 50)
print('MODEL EVALUATION SUMMARY')
print('=' * 50)
print(f'Test samples:          {len(all_preds)}')
print(f"MAE:                   {metrics['mae']:.2f}%")
print(f"RMSE:                  {metrics['rmse']:.2f}%")
print(f"Bias:                  {metrics['bias']:.2f}%")
print(f"Directional Accuracy:  {metrics['directional_accuracy']:.2f}")
print(f"Best Signal Accuracy:  {best_row['accuracy']:.2f} at +/-{best_row['threshold']}%")
print('=' * 50)
